# Alice & Bob Resource Estimator Tutorial

This Jupyter notebook is a tutorial for the **Alice & Bob Resource Estimator**, a tool designed to estimate the physical resources required to execute quantum algorithms on Alice & Bob hardware.

The tutorial introduces the estimator’s concepts, workflow, and usage through concrete examples, with a focus on elliptic curve cryptography (ECC).

# Table of Contents
1. [Prerequisites](#prerequisites)
2. [What Is the Alice & Bob Resource Estimator?](#what-is-the-alice--bob-resource-estimator)
3. [Installation](#installation)
4. [Examples](#examples)
   1. [ECC Based on arXiv:2302.06639](#ecc-based-on-arxiv230206639)
   2. [ECC Based on Logical Counts](#ecc-based-on-logical-counts)
   3. [ECC Based on Qualtran Implementation](#ecc-based-on-qualtran-implementation)
   4. [Estimation from a Q# File](#estimation-from-a-q-file)


## Prerequisites

The Alice & Bob Resource Estimator provides a Python interface, and no knowledge of Rust is required. However, readers are expected to have the following background:

- Familiarity with basic Python syntax
- A basic understanding of quantum circuits and common quantum gates
- A basic understanding of cat qubits
- Some familiarity with quantum error correction (QEC)

While **prior knowledge of elliptic curve cryptography** is helpful for interpreting the example results, it is **not required** to use the resource estimator itself.


## What Is the Alice & Bob Resource Estimator?

The [Microsoft Azure Quantum Resource Estimator (Aanb_estimator)](https://arxiv.org/abs/2311.05801) introduces a layered framework that connects high-level quantum programs to physical resource estimates through an intermediate quantum instruction set architecture (ISA).

The **Alice & Bob Resource Estimator** builds on this framework by specializing the hardware and error-correction layers for Alice & Bob’s **cat-qubit architecture**. Given a quantum algorithm expressed in a high-level programming language, the estimator:

1. Compiles the program to a quantum ISA,
2. Applies architecture-specific overhead models,
3. Produces detailed physical resource estimates.

These overhead models account for factors such as routing, quantum error correction, and magic-state factory implementations.


## Installation

This repository includes a `pixi.toml` file to enable reproducible environments.

To install the required environment and the Python extension, run the following commands:

To run the setup commands, you’ll need Pixi installed on your machine. This can be easily done by following the installation guide [here](https://pixi.prefix.dev/latest/installation/). Pixi will create and manage the project environment for you, including installing Python, Rust, and any required dependencies defined in the project configuration. In addition, you’ll need a working native build toolchain for your operating system (Xcode Command Line Tools on macOS, build-essential on Linux, or Microsoft C++ Build Tools on Windows), since maturin develop compiles Rust extensions locally.


In [4]:
#!pixi install
# You might also need to run the following command to build the rust dependencies
#!pixi run maturin develop --uv

Now you can import the resource estimator just like any other Python package.

In [5]:
# Import the Rust functionality as a Python module
import anb_estimator

# Examples
## ECC based on arXiv:2302.06639
The resource estimator includes a built-in example for computing elliptic curve discrete logarithms, based on the methodology described in [arXiv:2302.06639](https://arxiv.org/abs/2302.06639). The relevant function is documented below.

To run the example, it suffices to specify a bit size and a window size, which is used to perform windowed arithmetic.

In [6]:
elliptic_curve, _ = anb_estimator.estimate_ecc_example(256, 18, True)  # type: ignore
print("\n=== Elliptic curve cryptography example ===")
print(elliptic_curve)


=== Elliptic curve cryptography example ===
Estimates(physical_qubits=115203, runtime_seconds=27265.399421, runtime_hours=7.573722061388889, total_error=0.23104434880850558, code_distance=13, code_alpha2=19.0, factories=61, factories_distance=9, factories_alpha2=12.83, factory_fraction_percent=4.500750848502209, factory_fraction=0.04500750848502209)


The output summarizes the physical resource estimates produced by the Alice & Bob Resource Estimator for the elliptic curve cryptography example. It reports the total number of physical qubits required, the estimated runtime, and the resulting total logical error. Additional details describe the chosen quantum error-correcting code parameters, including the code distance and corresponding cat-state size, as well as the configuration of magic-state factories. In particular, the number of factories, their code distance, and the fraction of resources they consume provide insight into the overhead associated with fault-tolerant gate synthesis in this setting.

The parameters can be easily accessed as class variables as seen below 

In [7]:
code_distance = elliptic_curve.code_distance
print(f"Code distance: {code_distance}")
# Same for other parameters

Code distance: 13


## ECC based on logical counts

The same example can be executed in an equivalent manner using the more general interface, which accepts the tuple ``(#Qubits, #CX, #CCX)`` as input. The corresponding function docstring is shown below.

To obtain the logical resource counts, we reproduce the calculations described in [arXiv:2302.06639](https://arxiv.org/abs/2302.06639). As these calculations are identical to those used in the previous example, the resulting estimates are expected to match exactly. The only difference is in how the logical quantities are provided to the resource estimator: in the previous example, they are computed internally from the specified bit size and window size, whereas here the same logical counts are computed a priori and passed directly as input.

In [16]:
import math

bit_size = 256
window_size = 18

qubit_count = 9 * bit_size + window_size + 4

# Asymptotic gate counts, arXiv:2302.06639 (p. 21, app C.10)
cx_count = math.ceil(448 * bit_size**3 / window_size)
ccx_count = math.ceil(348 * bit_size**3 / window_size)

elliptic_curve_logic = anb_estimator.estimate_logical_counts(  # type: ignore
    qubit_count,
    cx_count,
    ccx_count,
    frontier=False,
    error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0),
)

In [15]:
print("\n=== Logical_estimates example ===")
print(elliptic_curve_logic)


=== Logical_estimates example ===
Estimates(physical_qubits=115203, runtime_seconds=27265.399421, runtime_hours=7.573722061388889, total_error=0.23104434880850558, code_distance=13, code_alpha2=19.0, factories=61, factories_distance=9, factories_alpha2=12.83, factory_fraction_percent=4.500750848502209, factory_fraction=0.04500750848502209)


## ECC based on Qualtran implementation
A third option is to use [Qualtran](https://qualtran.readthedocs.io/en/latest/index.html). Qualtran then produces an estimate of the logical resource counts, which can subsequently be used as input to the resource estimator.

The resource estimation can then be run by using the function ``estimate_from_qualtran`` as shown below

The cool thing about Qualtran is that with this interface we can easily perform a resource estimation for the large number of algorithms that are contained in the Qualtran library. Here is another example for the QFT on a register of 100 qubits.

In [10]:
from qualtran.bloqs.qft import QFTTextBook

qft_text_book = QFTTextBook(100)


result_qft = anb_estimator.estimate_from_qualtran(
    qft_text_book, frontier=False, error_total=None, error_budget=(0.333 * 0.5, 0.333 * 0.5, 0.0)
)

print(result_qft)

Estimates(physical_qubits=3205, runtime_seconds=0.1749825, runtime_hours=4.8606250000000006e-05, total_error=0.11649913091446393, code_distance=7, code_alpha2=11.0, factories=4, factories_distance=5, factories_alpha2=7.15, factory_fraction_percent=5.61622464898596, factory_fraction=0.056162246489859596)


## Estimation from a Q# file
The last option is to specify an algorithm in a Q# file and pass it as an input to the resource estimator. Although we want to concentrate our focus on using Qualtran, currently a lot of quantum algorithms are implemented in Q# so this functionality is still extremely valuable. The docstring can be seen below.

To perform the optimization we simply have to provide the path to the Q# file we are interested in. An example is given in ``qsharp/Adder.rs``.

In [11]:
filename = "../qsharp/Adder.qs"
single_qsharp, frontier, counts = anb_estimator.estimate_qsharp_file(  # type: ignore
    filename, frontier=True, error_total=None, error_budget=(0.0005, 0.0005, 0.0)
)
print("\n=== Q# example ===")
print(single_qsharp)


=== Q# example ===
Estimates(physical_qubits=13419, runtime_seconds=0.0132975, runtime_hours=3.69375e-06, total_error=0.00019340737414084432, code_distance=9, code_alpha2=14.0, factories=2, factories_distance=5, factories_alpha2=8.18, factory_fraction_percent=0.670690811535882, factory_fraction=0.00670690811535882)
